# 회계 분개장 오류 검증

계정과목·거래처·부서 기준정보와 회계 검증 규칙을 이용해
분개장의 입력 오류를 탐지하고 정답표와 비교해 검증 성능을 확인한다.

## 1. 기준정보와 검증 대상 데이터 불러오기

계정과목·거래처·부서 기준표와 오류가 포함된 분개장을 불러온다.
또한 검증 결과의 정확도를 평가하기 위해 오류 정답표를 함께 불러온다.

In [1]:
# 표 형태의 데이터를 처리하기 위한 라이브러리
import pandas as pd

# 운영체제와 관계없이 파일 경로를 안전하게 관리하기 위한 라이브러리
from pathlib import Path


# 현재 노트북이 실행되는 위치 확인
current_path = Path.cwd()

# 실행 위치가 notebooks 폴더라면 상위 폴더를 프로젝트 루트로 설정
if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

# 계정과목 기준표 파일 경로
accounts_path = (
    project_root
    / "data"
    / "master"
    / "accounts.csv"
)

# 거래처 기준표 파일 경로
vendors_path = (
    project_root
    / "data"
    / "master"
    / "vendors.csv"
)

# 부서 기준표 파일 경로
departments_path = (
    project_root
    / "data"
    / "master"
    / "departments.csv"
)

# 오류가 포함된 검증 대상 분개장 경로
journal_path = (
    project_root
    / "data"
    / "raw"
    / "journal_with_errors.xlsx"
)

# 03번 노트북에서 만든 오류 정답표 경로
expected_path = (
    project_root
    / "data"
    / "expected"
    / "injected_errors.csv"
)

# 각 파일을 데이터프레임으로 불러오기
accounts = pd.read_csv(accounts_path)
vendors = pd.read_csv(vendors_path)
departments = pd.read_csv(departments_path)

journal = pd.read_excel(
    journal_path,
    sheet_name="분개장"
)

expected_errors = pd.read_csv(expected_path)

# 파일별 데이터 개수 확인
print("계정과목 수:", len(accounts))
print("거래처 수:", len(vendors))
print("부서 수:", len(departments))
print("검증 대상 거래 수:", len(journal))
print("오류 정답 수:", len(expected_errors))

계정과목 수: 22
거래처 수: 12
부서 수: 7
검증 대상 거래 수: 10
오류 정답 수: 10


## 2. 유효 코드와 검증 기간 설정

마스터 데이터에서 현재 사용할 수 있는 계정과목·거래처·부서 코드를 추출한다.
발견한 오류를 동일한 형식으로 기록하기 위한 함수를 정의한다.

In [2]:
# 계정과목 코드를 숫자형으로 통일하고 중복이 없는 집합으로 변환
valid_account_codes = set(
    pd.to_numeric(
        accounts["account_code"],
        errors="coerce"
    )
    .dropna()
    .astype(int)
)

# 사용 상태가 Y인 거래처 코드만 추출
valid_partner_codes = set(
    vendors.loc[
        vendors["is_active"] == "Y",
        "partner_code"
    ]
    .dropna()
    .astype(str)
)

# 사용 상태가 Y인 부서 코드만 추출
valid_department_codes = set(
    departments.loc[
        departments["is_active"] == "Y",
        "department_code"
    ]
    .dropna()
    .astype(str)
)

# 이 프로젝트에서 정상으로 인정할 거래 기간 설정
period_start = pd.Timestamp("2026-08-01")
period_end = pd.Timestamp("2026-09-01")

# 앞으로 발견한 오류를 저장할 빈 목록
detected_errors = []

# 발견한 오류를 항상 동일한 열 구조로 저장하는 함수
def add_error(voucher_id, column, error_type, detail):
    detected_errors.append(
        {
            "voucher_id": voucher_id,
            "column": column,
            "error_type": error_type,
            "detail": detail
        }
    )

# 준비된 검증 기준 확인
print("유효 계정과목 수:", len(valid_account_codes))
print("유효 거래처 수:", len(valid_partner_codes))
print("유효 부서 수:", len(valid_department_codes))
print("정상 거래 시작일:", period_start.date())
print("정상 거래 종료 기준일:", period_end.date())
print("현재 탐지 오류 수:", len(detected_errors))

유효 계정과목 수: 22
유효 거래처 수: 12
유효 부서 수: 7
정상 거래 시작일: 2026-08-01
정상 거래 종료 기준일: 2026-09-01
현재 탐지 오류 수: 0


## 3. 필수값과 마스터 코드 검사

분개장의 필수 항목이 비어 있는지 확인한다.
입력된 계정과목·거래처·부서 코드가 각 마스터 데이터에 존재하는지도 검사한다.

In [3]:
# 반드시 값이 있어야 하는 필수 열
required_columns = [
    "voucher_id",
    "transaction_date",
    "department_code",
    "partner_code",
    "evidence_type",
    "evidence_no",
    "description"
]

# 계정과목 코드가 입력되는 열
account_columns = [
    "debit_account_1",
    "debit_account_2",
    "credit_account_1",
    "credit_account_2"
]


# 결측값과 빈 문자열을 함께 판별하는 함수
def is_missing(value):
    return pd.isna(value) or str(value).strip() == ""


# 분개장의 거래를 한 행씩 반복해 검사
for _, row in journal.iterrows():
    voucher_id = row["voucher_id"]

    # 필수 열의 값이 비어 있는지 검사
    for column in required_columns:
        if is_missing(row[column]):
            add_error(
                voucher_id=voucher_id,
                column=column,
                error_type="필수값 누락",
                detail=f"{column} 값 누락"
            )

    # 거래처 코드가 입력되어 있을 때만 기준표와 비교
    if not is_missing(row["partner_code"]):
        partner_code = str(row["partner_code"])

        if partner_code not in valid_partner_codes:
            add_error(
                voucher_id=voucher_id,
                column="partner_code",
                error_type="존재하지 않는 거래처",
                detail=f"{partner_code} 거래처 기준표에 없음"
            )

    # 부서 코드가 입력되어 있을 때만 기준표와 비교
    if not is_missing(row["department_code"]):
        department_code = str(row["department_code"])

        if department_code not in valid_department_codes:
            add_error(
                voucher_id=voucher_id,
                column="department_code",
                error_type="존재하지 않는 부서",
                detail=f"{department_code} 부서 기준표에 없음"
            )

    # 차변과 대변의 모든 계정과목 코드 검사
    for column in account_columns:
        account_value = row[column]

        # 두 번째 계정과목처럼 사용하지 않은 빈칸은 검사하지 않음
        if is_missing(account_value):
            continue

        # 엑셀에서 불러온 계정코드를 비교 가능한 숫자로 변환
        numeric_account = pd.to_numeric(
            account_value,
            errors="coerce"
        )

        # 숫자로 변환할 수 없거나 기준표에 없으면 오류로 기록
        if (
            pd.isna(numeric_account)
            or int(numeric_account) not in valid_account_codes
        ):
            add_error(
                voucher_id=voucher_id,
                column=column,
                error_type="존재하지 않는 계정과목",
                detail=f"{account_value} 계정과목 기준표에 없음"
            )


# 현재까지 발견한 오류를 표로 변환해 확인
current_errors = pd.DataFrame(detected_errors)

print("현재까지 탐지한 오류 수:", len(current_errors))

current_errors

현재까지 탐지한 오류 수: 5


,voucher_id,column,error_type,detail
0,JV202608001,debit_account_1,존재하지 않는 계정과목,9999 계정과목 기준표에 없음
1,JV202608002,partner_code,존재하지 않는 거래처,V999 거래처 기준표에 없음
2,JV202608003,department_code,존재하지 않는 부서,D999 부서 기준표에 없음
3,JV202608006,evidence_no,필수값 누락,evidence_no 값 누락
4,JV202608009,description,필수값 누락,description 값 누락


## 4. 회계금액·부가세·증빙·거래기간 검사

차변과 대변의 합계, 전표 총금액, 부가세 계산 결과를 검사한다.
중복 증빙번호와 회계기간을 벗어난 거래도 함께 탐지한다.

In [4]:
# 중복된 증빙번호 중 두 번째 이후에 등장한 거래만 True로 표시
# 증빙번호가 비어 있는 경우는 중복 검사에서 제외
duplicate_evidence_mask = (
    journal["evidence_no"].notna()
    & journal["evidence_no"].duplicated(keep="first")
)


# 분개장의 거래를 한 행씩 검사
for row_index, row in journal.iterrows():
    voucher_id = row["voucher_id"]

    # 비어 있는 금액은 0으로 처리하고 차변 합계 계산
    debit_total = (
        (0 if pd.isna(row["debit_amount_1"]) else row["debit_amount_1"])
        + (0 if pd.isna(row["debit_amount_2"]) else row["debit_amount_2"])
    )

    # 비어 있는 금액은 0으로 처리하고 대변 합계 계산
    credit_total = (
        (0 if pd.isna(row["credit_amount_1"]) else row["credit_amount_1"])
        + (0 if pd.isna(row["credit_amount_2"]) else row["credit_amount_2"])
    )

    # 차변 합계와 대변 합계가 일치하는지 검사
    if debit_total != credit_total:
        add_error(
            voucher_id=voucher_id,
            column="debit_amount_1",
            error_type="차변·대변 불일치",
            detail=(
                f"차변 {debit_total:,.0f}원 / "
                f"대변 {credit_total:,.0f}원"
            )
        )

    # 차변과 대변이 일치할 때만 전표 총금액과 비교
    elif row["total_amount"] != debit_total:
        add_error(
            voucher_id=voucher_id,
            column="total_amount",
            error_type="전표 합계 불일치",
            detail=(
                f"입력 {row['total_amount']:,.0f}원 / "
                f"분개 {debit_total:,.0f}원"
            )
        )

    # 공급가액이 있는 과세 거래만 부가세 검사
    if row["supply_amount"] > 0:
        expected_vat = round(
            row["supply_amount"] * 0.1
        )

        if row["vat_amount"] != expected_vat:
            add_error(
                voucher_id=voucher_id,
                column="vat_amount",
                error_type="부가세 계산 오류",
                detail=(
                    f"입력 {row['vat_amount']:,.0f}원 / "
                    f"예상 {expected_vat:,.0f}원"
                )
            )

    # 동일한 증빙번호가 앞선 거래에서 사용됐는지 검사
    if duplicate_evidence_mask.loc[row_index]:
        add_error(
            voucher_id=voucher_id,
            column="evidence_no",
            error_type="증빙번호 중복",
            detail=(
                f"{row['evidence_no']}: 중복 증빙"
            )
        )

    # 거래일자를 날짜 자료형으로 변환
    transaction_date = pd.to_datetime(
        row["transaction_date"],
        errors="coerce"
    )

    # 거래일자가 2026년 8월 범위에 포함되는지 검사
    if not pd.isna(transaction_date):
        if not (
            period_start
            <= transaction_date
            < period_end
        ):
            add_error(
                voucher_id=voucher_id,
                column="transaction_date",
                error_type="회계기간 이탈",
                detail=(
                    f"{transaction_date.date()}: "
                    "8월 회계기간 아님"
                )
            )


# 지금까지 발견한 전체 오류를 데이터프레임으로 변환
validation_result = pd.DataFrame(
    detected_errors
)

# 전표번호와 오류 유형 순서로 정렬
# 정렬 후 기존 행 번호를 0부터 다시 부여
validation_result = (
    validation_result
    .sort_values(
        by=[
            "voucher_id",
            "error_type"
        ]
    )
    .reset_index(drop=True)
)

print(
    "전체 탐지 오류 수:",
    len(validation_result)
)

validation_result

전체 탐지 오류 수: 10


,voucher_id,column,error_type,detail
0,JV202608001,debit_account_1,존재하지 않는 계정과목,9999 계정과목 기준표에 없음
1,JV202608002,partner_code,존재하지 않는 거래처,V999 거래처 기준표에 없음
2,JV202608003,department_code,존재하지 않는 부서,D999 부서 기준표에 없음
3,JV202608004,debit_amount_1,차변·대변 불일치,"차변 670,000원 / 대변 660,000원"
4,JV202608005,vat_amount,부가세 계산 오류,"입력 100,000원 / 예상 150,000원"
5,JV202608006,evidence_no,필수값 누락,evidence_no 값 누락
6,JV202608007,evidence_no,증빙번호 중복,TAX-202608-004: 중복 증빙
7,JV202608008,transaction_date,회계기간 이탈,2026-09-05: 8월 회계기간 아님
8,JV202608009,description,필수값 누락,description 값 누락
9,JV202608010,total_amount,전표 합계 불일치,"입력 5,400,000원 / 분개 5,500,000원"


## 5. 탐지 결과와 오류 정답표 비교

프로그램이 탐지한 오류를 03번 노트북에서 만든 정답표와 비교한다.
일치한 오류, 놓친 오류, 예상하지 못한 추가 탐지를 구분하고 성능을 계산한다.

In [5]:
# 정답표와 실제 탐지 결과를 전표번호·열·오류 유형 기준으로 비교
comparison = expected_errors.merge(
    validation_result,
    on=[
        "voucher_id",
        "column",
        "error_type"
    ],
    how="outer",
    indicator=True
)

# 병합 결과를 이해하기 쉬운 상태 이름으로 변경
comparison["comparison_status"] = comparison["_merge"].map(
    {
        "both": "정상 탐지",
        "left_only": "미탐지",
        "right_only": "추가 탐지"
    }
)

# 불필요해진 병합 상태 열 제거
comparison = comparison.drop(columns="_merge")

# 각 비교 상태의 건수 계산
status_summary = (
    comparison["comparison_status"]
    .value_counts()
    .rename_axis("comparison_status")
    .reset_index(name="count")
)

# 성능 계산에 필요한 건수
correct_count = (
    comparison["comparison_status"] == "정상 탐지"
).sum()

missed_count = (
    comparison["comparison_status"] == "미탐지"
).sum()

unexpected_count = (
    comparison["comparison_status"] == "추가 탐지"
).sum()

# 정답 중 프로그램이 실제로 찾아낸 비율
recall = (
    correct_count / (correct_count + missed_count)
    if correct_count + missed_count > 0
    else 0
)

# 프로그램이 탐지한 것 중 실제 정답이었던 비율
precision = (
    correct_count / (correct_count + unexpected_count)
    if correct_count + unexpected_count > 0
    else 0
)

print("정상 탐지:", correct_count)
print("미탐지:", missed_count)
print("추가 탐지:", unexpected_count)
print(f"재현율: {recall:.1%}")
print(f"정밀도: {precision:.1%}")

comparison

정상 탐지: 10
미탐지: 0
추가 탐지: 0
재현율: 100.0%
정밀도: 100.0%


,voucher_id,column,error_type,detail,comparison_status
0,JV202608001,debit_account_1,존재하지 않는 계정과목,9999 계정과목 기준표에 없음,정상 탐지
1,JV202608002,partner_code,존재하지 않는 거래처,V999 거래처 기준표에 없음,정상 탐지
2,JV202608003,department_code,존재하지 않는 부서,D999 부서 기준표에 없음,정상 탐지
3,JV202608004,debit_amount_1,차변·대변 불일치,"차변 670,000원 / 대변 660,000원",정상 탐지
4,JV202608005,vat_amount,부가세 계산 오류,"입력 100,000원 / 예상 150,000원",정상 탐지
5,JV202608006,evidence_no,필수값 누락,evidence_no 값 누락,정상 탐지
6,JV202608007,evidence_no,증빙번호 중복,TAX-202608-004: 중복 증빙,정상 탐지
7,JV202608008,transaction_date,회계기간 이탈,2026-09-05: 8월 회계기간 아님,정상 탐지
8,JV202608009,description,필수값 누락,description 값 누락,정상 탐지
9,JV202608010,total_amount,전표 합계 불일치,"입력 5,400,000원 / 분개 5,500,000원",정상 탐지


## 6. 오류 검증 결과와 성능 지표 저장

탐지한 오류 목록, 정답표 비교 결과, 검증 성능 요약을 각각 저장한다.
저장된 파일은 이후 ERP 입력 대상 분류와 BI 대시보드의 입력 데이터로 사용한다.

In [6]:
# 탐지 오류 목록을 저장할 경로
validation_result_path = (
    project_root
    / "data"
    / "processed"
    / "validation_result.csv"
)

# 정답표와 탐지 결과의 비교표를 저장할 경로
comparison_result_path = (
    project_root
    / "data"
    / "processed"
    / "validation_comparison.csv"
)

# 검증 성능 요약을 저장할 경로
validation_summary_path = (
    project_root
    / "outputs"
    / "validation_summary.csv"
)

# 대시보드와 포트폴리오에서 사용할 성능 요약표 생성
validation_summary = pd.DataFrame(
    [
        {
            "test_data": "합성 오류 분개장",
            "total_transactions": len(journal),
            "expected_errors": len(expected_errors),
            "detected_errors": len(validation_result),
            "correct_detections": correct_count,
            "missed_errors": missed_count,
            "unexpected_detections": unexpected_count,
            "precision": precision,
            "recall": recall
        }
    ]
)

# 탐지한 오류 전체 목록 저장
validation_result.to_csv(
    validation_result_path,
    index=False,
    encoding="utf-8-sig"
)

# 정답과 탐지 결과의 비교표 저장
comparison.to_csv(
    comparison_result_path,
    index=False,
    encoding="utf-8-sig"
)

# 검증 성능 요약표 저장
validation_summary.to_csv(
    validation_summary_path,
    index=False,
    encoding="utf-8-sig"
)

# 세 파일이 정상적으로 생성됐는지 확인
print("오류 목록 저장:", validation_result_path.exists())
print("비교 결과 저장:", comparison_result_path.exists())
print("성능 요약 저장:", validation_summary_path.exists())

validation_summary

오류 목록 저장: True
비교 결과 저장: True
성능 요약 저장: True


,test_data,total_transactions,expected_errors,detected_errors,correct_detections,missed_errors,unexpected_detections,precision,recall
0,합성 오류 분개장,10,10,10,10,0,0,1.0,1.0
